# Data Loading and Cleaning

This notebook loads the two raw SECOM files, joins sensors to labels by row order, sorts the units by time, and cleans the sensor table (drop sparse sensors, fill the remaining gaps with the median, drop constant sensors).

**Baseline note:** every cleaning rule here is learned from all 1,567 rows, so this is an exploration baseline, not a leakage-safe pipeline. The leakage-safe version learns its rules from the training period only.

In [1]:
import pandas as pd

## Raw files

The labels file is read first for a quick look at class balance. Pass = -1, fail = 1.

In [2]:
labels = pd.read_csv("../data/raw/secom_labels.data", sep=r"\s+", header=None)
print(labels.shape)
print(labels.head())

print(labels[0].value_counts())
print(labels.isna().sum().sum())

n_fail = (labels[0]==1).sum() # way 1 to calculate failure rates
n_total = len(labels)
print(n_fail / n_total)

print((labels[0] == 1).mean()) # way 2 to calculate failure rates

(1567, 2)
   0                    1
0 -1  19/07/2008 11:55:00
1 -1  19/07/2008 12:32:00
2  1  19/07/2008 13:17:00
3 -1  19/07/2008 14:43:00
4 -1  19/07/2008 15:22:00
0
-1    1463
 1     104
Name: count, dtype: int64
0
0.06636885768985322
0.06636885768985322


In [3]:
data = pd.read_csv("../data/raw/secom.data", sep=r"\s+", header=None)
print(data.shape)                  

n_missing = data.isna().sum().sum()
print(n_missing)                   
print(n_missing / data.size)       

(1567, 590)
41951
0.04537548808583821


## Building and joining the table

The sensor file and the labels file have no shared ID, so they are linked only by row order. They are joined first and sorted by timestamp afterwards, so every label stays attached to its own unit.

In [4]:
df = data.copy()
df.columns = [f"sensor_{i}" for i in range (df.shape[1])]
df["label"] = labels[0]
df["timestamp"] = pd.to_datetime(labels[1], format="%d/%m/%Y %H:%M:%S")
df = df.sort_values("timestamp").reset_index(drop=True)
print(df["timestamp"].is_monotonic_increasing)
print(df["timestamp"].dtype)                                # datetime64[...]
print(df["timestamp"].iloc[0], df["timestamp"].iloc[-1])    # 2008-07-19 ... 2008-10-17

True
datetime64[us]
2008-07-19 11:55:00 2008-10-17 06:07:00


## Missing values

Fraction of missing values per sensor, and how many sensors exceed the 20% threshold.

In [5]:
feature_cols = df.drop(columns=["label","timestamp"]).columns
missing_pct = df[feature_cols].isna().mean()
print(missing_pct.sort_values(ascending=False).head(10))

missing_pct_threshold = 0.2
print((missing_pct > missing_pct_threshold).sum())

sensor_292    0.911934
sensor_293    0.911934
sensor_158    0.911934
sensor_157    0.911934
sensor_492    0.855775
sensor_85     0.855775
sensor_358    0.855775
sensor_220    0.855775
sensor_244    0.649649
sensor_517    0.649649
dtype: float64
32


## Cleaning decisions

**Drop sensors with more than 20% missing values.** Filling them with a median or mean would make most of the column invented data, and a model could learn patterns that we made up.

**Fill the remaining gaps with the median.** The median is not pulled by outliers the way the mean is.

**Drop zero-variance (constant) sensors.** They cannot separate pass from fail, and they break calculations that divide by the standard deviation:
- standardizing gives 0 / 0 = NaN
- an SPC chart would have control limits of zero width
- Cp/Cpk divides by sigma, so it becomes infinite

In [6]:
high_missing = missing_pct[missing_pct > missing_pct_threshold].index.tolist()
df_clean = df.drop(columns=high_missing)
feature_cols = df_clean.drop(columns=["label", "timestamp"]).columns
df_clean[feature_cols] = df_clean[feature_cols].fillna(df_clean[feature_cols].median())

variances = df_clean[feature_cols].var()
zero_var = variances[variances == 0].index.tolist()
print(len(zero_var))
df_clean = df_clean.drop(columns=zero_var)
print(df_clean.shape)
df_clean.isna().sum().sum()

116
(1567, 444)


np.int64(0)

## What I learned

<!-- Draft: reword in your own words -->
- The two raw files are linked only by row order, so join first and sort after.
- Timestamps must be converted to real datetimes before sorting, because text sorts alphabetically.
- A passing check is not proof: `is_monotonic_increasing` was True on wrongly sorted text, and a correct shape hid 8,008 missing values.
- The median is used instead of the mean because outliers pull the mean.
- The rules in this notebook use all rows, so they leak future information. The redesign learns them from the training period only.